## Étape 0 : vérifications avant de commencer

Quatre vérifications à lancer dans l'ordre.

### 1. Installation de LangGraph

In [ ]:
!pip install langchain-community langchain langgraph snowflake-connector-python langchain-openai langchain-mcp-adapters rank_bm25

In [ ]:
%%sql -r dataframe_1
-- ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

### 2. Appel du LLM avec AI_COMPLETE

In [ ]:
import os
from pathlib import Path
from langchain_openai import ChatOpenAI

MODEL = "openai-gpt-5.4"


def make_llm(traceparent: str | None = None) -> ChatOpenAI:
    token = Path("/snowflake/session/token").read_text().strip()
    host = os.environ["SNOWFLAKE_HOST"]
    headers = {"X-Snowflake-Cross-Region": "ANY_REGION"}
    if traceparent:
        headers["traceparent"] = traceparent
    return ChatOpenAI(
        model=MODEL,
        base_url=f"https://{host}/api/v2/aigateways/SNOWFLAKE/v1",
        api_key=token,
        temperature=0,
        max_tokens=4096,
        default_headers=headers,
    )


llm = make_llm()
print("Modèle  :", MODEL)
print("Réponse :", llm.invoke("Réponds uniquement par le mot OK.").content)


### 4. Sortie structurée

C'est ce qui servira à l'orchestrateur pour choisir entre répondre, demander une précision ou escalader.

Résultat attendu : `issue = preciser` (le ticket ne donne ni route ni référence).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class Decision(BaseModel):
    raison: str = Field(description="Justification courte de la décision")
    issue: Literal["repondre", "preciser", "escalader"]


ticket = "Le prix affiché sur SpotOn ne correspond pas à celui attendu."

decision = llm.with_structured_output(Decision).invoke(
    "Tu traites un ticket de support. Choisis 'preciser' s'il manque la route, "
    "la date ou une référence de devis, 'escalader' si le cas dépasse le support, "
    "sinon 'repondre'.\n"
    f"Ticket : {ticket}"
)
print("issue  =", decision.issue)
print("raison =", decision.raison)

### 3. Graphe LangGraph minimal

Résultat attendu : `{'ticket': 'test', 'trace': ['intake']}`

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    ticket: str
    trace: list


def intake(state: State):
    return {"trace": state["trace"] + ["intake"]}


g = StateGraph(State)
g.add_node("intake", intake)
g.add_edge(START, "intake")
g.add_edge("intake", END)

app = g.compile()
print(app.invoke({"ticket": "test", "trace": []}))

### 4. Import d'un module de `src/`

Les deux premiers affichages montrent où le notebook s'exécute réellement. Résultat attendu à la fin : `Import OK : ok`

In [ ]:
import os
import sys

print("Dossier courant :", os.getcwd())
print("Contenu :", os.listdir("."))

parent = os.path.abspath("..")
print("Dossier parent  :", parent)
print("Contenu parent  :", os.listdir(parent))

if parent not in sys.path:
    sys.path.append(parent)

from src import state
print("Import OK :", state.TEST)